<a href="https://colab.research.google.com/github/jyryu3161/LAIDD_metabolicmodeling/blob/main/lec02_%E1%84%89%E1%85%B5%E1%86%AF%E1%84%89%E1%85%B3%E1%86%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Simulating with FBA

In [1]:
!pip install cobra

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 39.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 92.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 85.5 MB/s eta 0:00:00


Simulations using flux balance analysis can be solved using `Model.optimize()`. This will maximize or minimize (maximizing is the default) flux through the objective reactions.

In [2]:
from cobra.io import load_model
model = load_model("textbook")


In [ ]:
solution = model.optimize() # FBA 실행
print(solution)

<Solution 0.874 at 0x7dc18801f140>


In [3]:
from cobra.io import read_sbml_model, write_sbml_model
import logging

model = read_sbml_model("e_coli_core.xml")

In [4]:
print(len(model.reactions))
print(len(model.metabolites))
print(len(model.genes))

95
72
137


## Running FBA

In [5]:
solution = model.optimize()
print(solution)

<Solution 0.874 at 0x7a4638b49b20>


In [6]:
model.objective

The Model.optimize() function will return a Solution object. A solution object has several attributes:

 - `objective_value`: the objective value
 - `status`: the status from the linear programming solver
 - `fluxes`: a pandas series with flux indexed by reaction identifier. The flux for a reaction variable is the difference of the primal values for the forward and reverse reaction variables.
 - `shadow_prices`: a pandas series with shadow price indexed by the metabolite identifier.

For example, after the last call to `model.optimize()`, if the optimization succeeds it's status will be optimal. In case the model is infeasible an error is raised.

In [7]:
solution.objective_value

0.8739215069684295

The solvers that can be used with cobrapy are so fast that for many small to mid-size models computing the solution can be even faster than it takes to collect the values from the solver and convert to them python objects. With `model.optimize`, we gather values for all reactions and metabolites and that can take a significant amount of time if done repeatedly. If we are only interested in the flux value of a single reaction or the objective, it is faster to instead use `model.slim_optimize` which only does the optimization and returns the objective value leaving it up to you to fetch other values that you may need.

In [8]:
%%time
model.optimize().objective_value # 전체 flux 값

CPU times: user 2 ms, sys: 0 ns, total: 2 ms
Wall time: 2.13 ms


0.873921506968431

In [10]:
solution = model.optimize()
print(solution)
print(solution.fluxes)

for each_reaction in dict(solution.fluxes):
    print(each_reaction, solution.fluxes[each_reaction])
    break

<Solution 0.874 at 0x7a4638a31310>
PFK         7.477382
PFL         0.000000
PGI         4.860861
PGK       -16.023526
PGL         4.959985
             ...    
NADH16     38.534610
NADTRHD     0.000000
NH4t        4.765319
O2t        21.799493
PDH         9.282533
Name: fluxes, Length: 95, dtype: float64
PFK 7.477381962160284


In [11]:
%%time
model.slim_optimize() # 목적 함수의 값만 필요

CPU times: user 324 µs, sys: 44 µs, total: 368 µs
Wall time: 382 µs


0.873921506968431

In [12]:
solution = model.slim_optimize() # 목적 함수의 값만 필요

### Analyzing FBA solutions

Models solved using FBA can be further analyzed by using summary methods, which output printed text to give a quick representation of model behavior. Calling the summary method on the entire model displays information on the input and output behavior of the model, along with the optimized objective.

In [13]:
model.summary()

Metabolite,Reaction,Flux,C-Number,C-Flux
glc__D_e,EX_glc__D_e,10,6,100.00%
nh4_e,EX_nh4_e,4.765,0,0.00%
o2_e,EX_o2_e,21.8,0,0.00%
pi_e,EX_pi_e,3.215,0,0.00%
Metabolite,Reaction,Flux,C-Number,C-Flux
co2_e,EX_co2_e,-22.81,1,100.00%
h2o_e,EX_h2o_e,-29.18,0,0.00%
h_e,EX_h_e,-17.53,0,0.00%


In addition, the input-output behavior of individual metabolites can also be inspected using summary methods. For instance, the following commands can be used to examine the overall redox balance of the model

In [14]:
model.metabolites.nadh_c.summary()

Percent,Flux,Reaction,Definition
13.14%,5.064,AKGDH,akg_c + coa_c + nad_c --> co2_c + nadh_c + succoa_c
8.04%,3.1,BIOMASS_Ecoli_core_w_GAM,1.496 3pg_c + 3.7478 accoa_c + 59.81 atp_c + 0.361 e4p_c + 0.0709 f6p_c + 0.129 g3p_c + 0.205 g6p_c + 0.2557 gln__L_c + 4.9414 glu__L_c + 59.81 h2o_c + 3.547 nad_c + 13.0279 nadph_c + 1.7867 oaa_c + 0.5191 pep_c + 2.8328 pyr_c + 0.8977 r5p_c --> 59.81 adp_c + 4.1182 akg_c + 3.7478 coa_c + 59.81 h_c + 3.547 nadh_c + 13.0279 nadp_c + 59.81 pi_c
41.58%,16.02,GAPD,g3p_c + nad_c + pi_c <=> 13dpg_c + h_c + nadh_c
13.14%,5.064,MDH,mal__L_c + nad_c <=> h_c + nadh_c + oaa_c
24.09%,9.283,PDH,coa_c + nad_c + pyr_c --> accoa_c + co2_c + nadh_c
Percent,Flux,Reaction,Definition
100.00%,-38.53,NADH16,4.0 h_c + nadh_c + q8_c --> 3.0 h_e + nad_c + q8h2_c


Or to get a sense of the main energy production and consumption reactions

In [15]:
model.metabolites.atp_c.summary()

Percent,Flux,Reaction,Definition
66.58%,45.51,ATPS4r,adp_c + 4.0 h_e + pi_c <=> atp_c + h2o_c + 3.0 h_c
23.44%,16.02,PGK,3pg_c + atp_c <=> 13dpg_c + adp_c
2.57%,1.758,PYK,adp_c + h_c + pep_c --> atp_c + pyr_c
7.41%,5.064,SUCOAS,atp_c + coa_c + succ_c <=> adp_c + pi_c + succoa_c
Percent,Flux,Reaction,Definition
12.27%,-8.39,ATPM,atp_c + h2o_c --> adp_c + h_c + pi_c
76.46%,-52.27,BIOMASS_Ecoli_core_w_GAM,1.496 3pg_c + 3.7478 accoa_c + 59.81 atp_c + 0.361 e4p_c + 0.0709 f6p_c + 0.129 g3p_c + 0.205 g6p_c + 0.2557 gln__L_c + 4.9414 glu__L_c + 59.81 h2o_c + 3.547 nad_c + 13.0279 nadph_c + 1.7867 oaa_c + 0.5191 pep_c + 2.8328 pyr_c + 0.8977 r5p_c --> 59.81 adp_c + 4.1182 akg_c + 3.7478 coa_c + 59.81 h_c + 3.547 nadh_c + 13.0279 nadp_c + 59.81 pi_c
0.33%,-0.2235,GLNS,atp_c + glu__L_c + nh4_c --> adp_c + gln__L_c + h_c + pi_c
10.94%,-7.477,PFK,atp_c + f6p_c --> adp_c + fdp_c + h_c


## Changing the Objectives

The objective function is determined from the objective_coefficient attribute of the objective reaction(s). Generally, a "biomass" function which describes the composition of metabolites which make up a cell is used.

The objective function can be changed by assigning Model.objective, which can be a reaction object (or just it's name), or a `dict` of `{Reaction: objective_coefficient}`.

In [16]:
# change the objective to ATPM
model.objective = "ATPM"

# The upper bound should be 1000, so that we get
# the actual optimal value
model.reactions.get_by_id("ATPM").upper_bound = 1000.


In [17]:
model.optimize().objective_value

174.99999999999807

We can also have more complicated objectives including quadratic terms.

## Running FVA

FBA will not give always give unique solution, because multiple flux states can achieve the same optimum. FVA (or flux variability analysis) finds the ranges of each metabolic flux at the optimum.

In [24]:
from cobra.flux_analysis import flux_variability_analysis
from cobra.io import read_sbml_model, write_sbml_model
import logging
import cobra

model = read_sbml_model("e_coli_core.xml")


In [25]:
flux_variability_analysis(model, model.reactions[:10]) # biomass max 0.87

,minimum,maximum
PFK,7.477382e+00,7.477382e+00
PFL,0.000000e+00,1.818351e-13
PGI,4.860861e+00,4.860861e+00
PGK,-1.602353e+01,-1.602353e+01
PGL,4.959985e+00,4.959985e+00
ACALD,-4.599139e-14,0.000000e+00
AKGt2r,-2.064529e-14,0.000000e+00
PGM,-1.471614e+01,-1.471614e+01
PIt2r,3.214895e+00,3.214895e+00
ALCD2x,-4.005702e-14,0.000000e+00


Setting parameter `fraction_of_optimium=0.90` would give the flux ranges for reactions at 90% optimality.

In [26]:
cobra.flux_analysis.flux_variability_analysis(
    model, model.reactions[:10], fraction_of_optimum=0.9) # 0.87 * 0.9 =

,minimum,maximum
PFK,1.171706,25.290644
PFL,0.000000,11.322324
PGI,-14.299039,9.838761
PGK,-17.909169,-9.863235
PGL,0.000000,24.137801
ACALD,-2.542370,0.000000
AKGt2r,-1.430083,0.000000
PGM,-16.732521,-8.686588
PIt2r,2.893406,3.214895
ALCD2x,-2.214323,0.000000


The standard FVA may contain loops, i.e. high absolute flux values that only can be high if they are allowed to participate in loops (a mathematical artifact that cannot happen in vivo). Use the `loopless` argument to avoid such loops. Below, we can see that FRD7 and SUCDi reactions can participate in loops but that this is avoided when using the looplesss FVA.

## Running pFBA

Parsimonious FBA (often written pFBA) finds a flux distribution which gives the optimal growth rate, but minimizes the total sum of flux. This involves solving two sequential linear programs, but is handled transparently by cobrapy. For more details on pFBA, please see [Lewis et al. (2010)](http://dx.doi.org/10.1038/msb.2010.47).

In [28]:
from cobra.io import read_sbml_model, write_sbml_model
import cobra
import logging

# objective value = 0.87 만족 하면서, 나머지 반응식들의 flux 합은 최소화한다.
objective_reaction = 'BIOMASS_Ecoli_core_w_GAM'
model = read_sbml_model("e_coli_core.xml")

fba_solution = model.optimize()
pfba_solution = cobra.flux_analysis.pfba(model)

print(pfba_solution.fluxes[objective_reaction])

0.8739215069684311


These functions will likely give completely different objective values, because the objective value shown from pFBA is defined as `sum(abs(pfba_solution.fluxes.values))`, while the objective value for standard FBA is defined as the weighted flux through the reactions being optimized for (e.g. `fba_solution.fluxes["Biomass_Ecoli_core"]`).

Both pFBA and FBA should return identical results within solver tolerances for the objective being optimized. E.g. for a FBA problem which maximizes the reaction `Biomass_Ecoli_core`:

In [29]:
abs(fba_solution.fluxes["BIOMASS_Ecoli_core_w_GAM"] - pfba_solution.fluxes[
    "BIOMASS_Ecoli_core_w_GAM"])

np.float64(1.5543122344752192e-15)

In [30]:
pfba_solution

,fluxes,reduced_costs
PFK,7.477382,-2.000000
PFL,0.000000,1.733333
PGI,4.860861,-2.000000
PGK,-16.023526,2.000000
PGL,4.959985,-2.000000
...,...,...
NADH16,38.534610,-2.000000
NADTRHD,0.000000,1.422222
NH4t,4.765319,-2.000000
O2t,21.799493,-2.000000


In [33]:
print(pfba_solution.fluxes[objective_reaction])

0.8739215069684311


In [35]:
for each_reaction in dict(fba_solution.fluxes):
    print(each_reaction, fba_solution.fluxes[each_reaction], pfba_solution.fluxes[each_reaction])


PFK 7.47738196216029 7.477381962160289
PFL 0.0 0.0
PGI 4.860861146496837 4.860861146496826
PGK -16.023526143167615 -16.02352614316761
PGL 4.959984944574638 4.959984944574647
ACALD 0.0 0.0
AKGt2r 0.0 0.0
PGM -14.71613956874284 -14.716139568742836
PIt2r 3.21489504768473 3.2148950476847618
ALCD2x 0.0 0.0
ACALDt 0.0 0.0
ACKr 1.0297466822596379e-15 3.642474998683731e-14
PPC 2.5043094703687303 2.5043094703687423
ACONTa 6.007249575350356 6.007249575350363
ACONTb 6.007249575350356 6.007249575350364
ATPM 8.39 8.39
PPCK 0.0 0.0
ACt2r 1.0297466822596379e-15 3.642474998683731e-14
PPS 0.0 0.0
ADK1 0.0 0.0
AKGDH 5.064375661482119 5.064375661482116
ATPS4r 45.51400977451756 45.51400977451754
PTAr -5.196301174036498e-15 -3.642474998683731e-14
PYK 1.7581774441067952 1.758177444106781
BIOMASS_Ecoli_core_w_GAM 0.8739215069684295 0.8739215069684311
PYRt2 -7.597785447006458e-16 0.0
CO2t -22.809833310205004 -22.809833310204997
RPE 2.678481850507515 2.678481850507522
CS 6.007249575350356 6.007249575350363
RPI